# Bonus – Python without the GIL

In the warm-up we saw that threads bring **no speed-up** for CPU-bound Python code, because of the
Global Interpreter Lock (GIL). Since Python 3.13 there is a separate **free-threaded build**
(`python3.13t`, `python3.14t`) that runs without the GIL (PEP 703).

You cannot switch the GIL off in the normal build – you need that separate build. In this notebook we
install it next to our normal Python and run the **same benchmark** three times:

| Run | Python | GIL |
|---|---|---|
| 1 | 3.12, normal build (our kernel) | always on |
| 2 | 3.14t, free-threaded build | switched back on with `PYTHON_GIL=1` |
| 3 | 3.14t, free-threaded build | off (the default) |

⏱ About 5 minutes. The first run downloads ~50 MB.

## 1. Which Python is this notebook running on?

In [ ]:
import sys, os, json, subprocess

print(sys.version)
is_gil_enabled = getattr(sys, "_is_gil_enabled", None)       # exists from Python 3.13 on
print("GIL enabled:", is_gil_enabled() if is_gil_enabled else "yes – normal build, the GIL cannot be switched off")
print("CPU cores:  ", os.cpu_count(), "(as the operating system sees them)")

# In the cloud, one "vCPU" is often one hyperthread – two vCPUs can share one physical core.
if sys.platform.startswith("linux"):
    info = subprocess.run(["lscpu"], capture_output=True, text=True).stdout
    for line in info.splitlines():
        if line.startswith(("Thread(s) per core", "Core(s) per socket", "Socket(s)")):
            print(line)

## 2. Install the free-threaded build

We use **uv**, a fast Python installer. It puts Python 3.14t into your home folder;
nothing in the course environment changes.

In [ ]:
def sh(*cmd, **kw):
    """Run a command and return its output – or show its error message."""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, cmd))} failed:\n{r.stderr[-2000:]}")
    return r.stdout.strip()

sh(sys.executable, "-m", "pip", "install", "-q", "uv")
sh(sys.executable, "-m", "uv", "python", "install", "3.14t")
FT = sh(sys.executable, "-m", "uv", "python", "find", "3.14t")

print("Free-threaded Python:", FT)
print(sh(FT, "-VV"))

## 3. The benchmark

The same CPU-bound task as in the warm-up: 4 jobs, first one after another (serial), then in 4 threads.
Each measurement is repeated 3 times and the fastest run counts, because other programs on the machine
(VS Code, other notebooks) can disturb a single measurement.
The script writes itself to a file, because the GIL setting has to be chosen **when Python starts** –
we cannot change it inside this running notebook.

In [ ]:
%%writefile /tmp/gil_demo.py
import sys, time, json
from concurrent.futures import ThreadPoolExecutor

def cpu_heavy(n):
    total = 0
    for i in range(n):
        total += i
    return total

N, JOBS, REPEATS = 10**7, 4, 3

def best_of(fn):
    # run fn several times and keep the fastest run – removes noise from other programs
    times = []
    for _ in range(REPEATS):
        t = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t)
    return min(times)

serial = best_of(lambda: [cpu_heavy(N) for _ in range(JOBS)])
with ThreadPoolExecutor(JOBS) as ex:
    threads = best_of(lambda: list(ex.map(cpu_heavy, [N] * JOBS)))

gil = sys._is_gil_enabled() if hasattr(sys, "_is_gil_enabled") else True
print(json.dumps({"python": sys.version.split()[0], "gil": gil,
                  "serial_s": round(serial, 2), "threads_s": round(threads, 2),
                  "speedup": round(serial / threads, 2)}))

## 4. Run it three times

In [ ]:
def run(python, gil=None):
    env = dict(os.environ)
    if gil is not None:
        env["PYTHON_GIL"] = str(gil)          # 1 = GIL on, 0 = GIL off (free-threaded build only)
    return json.loads(sh(python, "/tmp/gil_demo.py", env=env).splitlines()[-1])

runs = {
    f"{sys.version.split()[0]} normal build": run(sys.executable),
    "3.14t, PYTHON_GIL=1":      run(FT, gil=1),
    "3.14t, GIL off (default)": run(FT, gil=0),
}

print(f"{'Run':<26}{'GIL':>6}{'serial':>9}{'threads':>9}{'speed-up':>10}")
for name, r in runs.items():
    print(f"{name:<26}{str(r['gil']):>6}{r['serial_s']:>8.2f}s{r['threads_s']:>8.2f}s{r['speedup']:>9.1f}x")

## 5. Questions

1. In the first two runs, 4 threads are no faster than 1 thread (speed-up column ≈ 1.0x). Why?
   *(The speed-up compares threads with serial **within one run**, not one run with another.)*
2. What limits the speed-up of run 3? Look at `JOBS` in the script and at section 1: how many
   **physical** cores does your machine have? (If *Thread(s) per core* is 2, two "CPUs" share one core –
   and threads doing pure computation barely gain anything.)
3. Compare the **serial** times of run 1 and run 3. Two things differ: the Python version and the
   GIL. Which one explains the difference? (Hint: Python 3.11+ made the interpreter itself much faster.)
   Why could removing the GIL also *cost* something?
4. Polars, DuckDB and Spark were fast in the main lab – with the normal Python build. How do they
   avoid the GIL problem without a free-threaded Python?

### Good to know
- Python 3.13 called free-threading *experimental*; from **3.14** it is officially supported, and the
  single-thread slowdown is much smaller.
- Not every library is ready yet. If you import a C extension that is not marked as free-threading-safe,
  Python switches the GIL back on and prints a warning.
- Check any time with `sys._is_gil_enabled()`.